# Dipole Angular Separation — Histograms per Deep Field

This notebook studies the distribution of the **angular separation** `r:dipoleLength`
(the centre-to-centre distance between the positive and negative lobes of each dipole
alert, expressed in arcseconds or pixels depending on the AP pipeline version)
for the six LSST Deep Drilling Fields.

## Layout

A single 2×3 figure (one subplot per DDF) shows **log-binned histograms**
of `r:dipoleLength`, stacked by band in the standard ugrizy order and colour code.
Log binning is motivated by the broad dynamic range of separations:
some dipoles have sub-pixel separations while others exceed 10× that value.

## Physical expectation

* **ECDFS** (δ ≈ −27.8°) is close to the zenith at Cerro Pachón (φ ≈ −30.2°), so
  observations near transit have small zenith angles and therefore small DCR
  displacements → smaller separations expected.
* **COSMOS** (δ ≈ +2.2°) and **M49** (δ ≈ +8.0°) are far from the zenith at
  transit, and are observed at higher airmass → larger DCR amplitudes
  → larger separations expected.
* **EDFS-a/b** (δ ≈ −49°) are south of the zenith and also observed at moderate
  to high airmass.

## Also kept: full observing-geometry table

The notebook recomputes the complete geometry (η, H, azimuth, zenith, sin z,
airmass) inherited from `05b_dipole_parallacticcorr.ipynb` and `06_checkdirections_astroplan.ipynb`,
stored in `df_all`, for downstream use.


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-05-31
- last update : 2026-05-31

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

from astropy.time import Time
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
import astropy.units as u

warnings.filterwarnings("ignore")
print(f"pandas  {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → %matplotlib widget")
except ImportError:
    %matplotlib inline
    print("ipympl not found → %matplotlib inline")

In [ ]:
# ── I/O paths ─────────────────────────────────────────────────────────────────
DIR_DATA_IN = "data_DIPOLES_01c"
NB_TAG = "DIPOLES_08"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Input  : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figures: {os.path.abspath(DIR_FIGS)}")

# ── Rubin/LSST – Cerro Pachón ─────────────────────────────────────────────────
RUBIN_LAT_DEG = -30.244728
RUBIN_LON_DEG = -70.749417
RUBIN_HEIGHT_M = 2647.0
RUBIN_LOCATION = EarthLocation(
    lat=RUBIN_LAT_DEG * u.deg,
    lon=RUBIN_LON_DEG * u.deg,
    height=RUBIN_HEIGHT_M * u.m,
)
print(f"Observatory: lat={RUBIN_LAT_DEG}°  lon={RUBIN_LON_DEG}°  h={RUBIN_HEIGHT_M} m")

# ── LSST Deep Drilling Fields ─────────────────────────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}
DDF_NAMES = list(DEEP_FIELDS.keys())  # fixed order for 2×3 grid

# ── Band colours (LSST ugrizy) ────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
BAND_ORDER = list("ugrizy")

# ── Matplotlib defaults ───────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure as PDF + PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. Observing-geometry helpers

Reused verbatim from `05b_dipole_parallacticcorr.ipynb` and
`06_checkdirections_astroplan.ipynb`.  The main workhorse is
`compute_observing_geometry` which returns η, H, azimuth, zenith,
sin z, and airmass for an array of (RA, Dec, MJD) triples.

The parallactic angle formula:
$$H = \mathrm{LST} - \alpha, \qquad
\eta = \arctan2\!\left(\sin H,\;\tan\phi\cos\delta - \sin\delta\cos H\right)$$

In [ ]:
def calculate_parallactic_angle_fromHA(ha, coords, location):
    """
    Parallactic angle η from a precomputed hour-angle array.

    Parameters
    ----------
    ha       : astropy Angle (degrees)
    coords   : SkyCoord of the target
    location : EarthLocation of the observatory

    Returns
    -------
    η in degrees, range [−180°, +180°]
    """
    ha_rad = ha.to(u.rad).value
    phi = location.lat.to(u.rad).value
    dec_rad = coords.dec.to(u.rad).value
    sinH = np.sin(ha_rad)
    cosH = np.cos(ha_rad)
    q = np.arctan2(sinH, np.tan(phi) * np.cos(dec_rad) - np.sin(dec_rad) * cosH)
    return np.degrees(q)


def sinz_vs_HA(HA_deg, coords, location):
    """
    sin z from the analytic formula:
    sin z = sqrt(1 - (sin φ sin δ + cos φ cos δ cos H)²)
    """
    lat_deg = location.lat.to(u.deg).value
    dec_deg = coords.dec.to(u.deg).value
    HA_valdeg = HA_deg.to(u.deg).value
    HA = np.deg2rad(HA_valdeg)
    dec = np.deg2rad(dec_deg)
    lat = np.deg2rad(lat_deg)
    cosz = np.sin(lat) * np.sin(dec) + np.cos(lat) * np.cos(dec) * np.cos(HA)
    return np.sqrt(1 - cosz**2)


print("Geometry helpers defined.")

In [ ]:
def compute_observing_geometry(
    ra_deg: np.ndarray,
    dec_deg: np.ndarray,
    mjd: np.ndarray,
    location: EarthLocation = RUBIN_LOCATION,
    batch_size: int = 500,
) -> pd.DataFrame:
    """
    Compute full observing geometry for a set of alerts.

    Parameters
    ----------
    ra_deg, dec_deg : array-like  – ICRS coordinates in degrees
    mjd             : array-like  – MJD TAI
    location        : EarthLocation
    batch_size      : int  – alerts per astropy call (speed/memory trade-off)

    Returns
    -------
    pd.DataFrame with columns:
        parallactic_angle_deg   float   −180 … +180°   (North = 0, CCW)
        hour_angle_hr           float   −12 … +12 h    (H = LST − RA)
        hour_angle_deg          float   −180 … +180°   (same × 15)
        azimuth_deg             float     0 … 360°     (North = 0, E = 90)
        altitude_deg            float     0 …  90°
        zenith_angle_deg        float     0 …  90°
        sin_zenith              float     0 … 1
        airmass                 float   ≥ 1             (≈ 1/cos z)
    """
    ra = np.asarray(ra_deg, dtype=float)
    dec = np.asarray(dec_deg, dtype=float)
    t = np.asarray(mjd, dtype=float)
    n = len(ra)

    para = np.full(n, np.nan)
    H_hr = np.full(n, np.nan)
    az = np.full(n, np.nan)
    alt = np.full(n, np.nan)
    za = np.full(n, np.nan)

    for i0 in range(0, n, batch_size):
        sl = slice(i0, min(i0 + batch_size, n))
        try:
            times = Time(t[sl], format="mjd", scale="tai").ut1
            coords = SkyCoord(ra=ra[sl] * u.deg, dec=dec[sl] * u.deg)

            lst = times.sidereal_time("apparent", longitude=location.lon)
            H_wrap = (lst - coords.ra).wrap_at(180 * u.deg)
            H_rad = H_wrap.to(u.rad).value
            H_hr[sl] = H_wrap.to(u.hourangle).value

            phi = location.lat.to(u.rad).value
            dec_rad = coords.dec.to(u.rad).value
            para[sl] = np.degrees(
                np.arctan2(
                    np.sin(H_rad),
                    np.tan(phi) * np.cos(dec_rad) - np.sin(dec_rad) * np.cos(H_rad),
                )
            )

            frame = AltAz(obstime=times, location=location)
            altaz = coords.transform_to(frame)
            alt[sl] = altaz.alt.deg
            az[sl] = altaz.az.deg
            za[sl] = 90.0 - altaz.alt.deg
        except Exception as exc:
            print(f"  [warning] batch {i0}–{i0 + batch_size}: {exc}")

    with np.errstate(divide="ignore", invalid="ignore"):
        airmass = np.where(za < 89.0, 1.0 / np.cos(np.radians(za)), np.nan)

    return pd.DataFrame(
        {
            "parallactic_angle_deg": para,
            "hour_angle_hr": H_hr,
            "hour_angle_deg": H_hr * 15.0,
            "azimuth_deg": az,
            "altitude_deg": alt,
            "zenith_angle_deg": za,
            "sin_zenith": np.sin(np.radians(za)),
            "airmass": airmass,
        }
    )


# Sanity check
test = compute_observing_geometry([150.1191], [2.2058], [60310.5])
print("Sanity check COSMOS MJD=60310.5:")
print(test.to_string(index=False))

## 3. Load dipole alerts

In [ ]:
ddf_alerts: dict[str, pd.DataFrame] = {}

for field_name in DDF_NAMES:
    pq = os.path.join(DIR_DATA_IN, f"{field_name}_alerts.parquet")
    if not os.path.exists(pq):
        print(f"[{field_name:12s}] parquet not found — skipping.")
        ddf_alerts[field_name] = pd.DataFrame()
        continue

    df = pd.read_parquet(pq)

    # Cast boolean isDipole
    if "r:isDipole" in df.columns:
        df["r:isDipole"] = (
            df["r:isDipole"]
            .map(
                lambda v: (
                    True
                    if str(v).strip().lower() in ("true", "1", "yes")
                    else False
                    if str(v).strip().lower() in ("false", "0", "no")
                    else pd.NA
                )
            )
            .astype("boolean")
        )

    for col in (
        "r:midpointMjdTai",
        "r:ra",
        "r:dec",
        "r:dipoleAngle",
        "r:dipoleLength",
        "r:dipoleChi2",
        "r:dipoleFluxDiff",
    ):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df_dip = (
        df[df["r:isDipole"].fillna(False).astype(bool)].copy()
        if "r:isDipole" in df.columns
        else pd.DataFrame()
    )
    df_dip["field"] = field_name
    ddf_alerts[field_name] = df_dip
    print(f"[{field_name:12s}] {len(df):7,} total  |  {len(df_dip):6,} dipoles")

print("\nLoad complete.")

## 4. Compute observing geometry

Full geometry (η, H, azimuth, zenith, sin z, airmass) plus signed and
folded dipole-angle differences are computed here and kept in `df_all`
for potential downstream use.

| Column | Definition |
|--------|------------|
| `parallactic_angle_deg` | η = arctan2(sin H, tan φ cos δ − sin δ cos H) |
| `hour_angle_hr` | H = LST − RA in hours |
| `hour_angle_deg` | H × 15 in degrees |
| `azimuth_deg` | CW from North |
| `zenith_angle_deg` | 90° − altitude |
| `sin_zenith` | sin(z), DCR amplitude proxy |
| `airmass` | 1/cos(z) |
| `delta_dipole_para` | r:dipoleAngle − η, wrapped to (−180°, +180°] |
| `delta_dipole_para_folded` | min(\|Δ\|, 180°−\|Δ\|) ∈ [0°, 90°] |

In [ ]:
frames: list[pd.DataFrame] = []

for field_name in DDF_NAMES:
    df_dip = ddf_alerts.get(field_name, pd.DataFrame())
    if df_dip.empty:
        print(f"[{field_name:12s}] no dipoles — skipping.")
        continue

    need = ["r:ra", "r:dec", "r:midpointMjdTai"]
    missing = [c for c in need if c not in df_dip.columns]
    if missing:
        print(f"[{field_name:12s}] missing {missing} — skipping.")
        continue

    mask = df_dip["r:ra"].notna() & df_dip["r:dec"].notna() & df_dip["r:midpointMjdTai"].notna()
    df_clean = df_dip[mask].copy().reset_index(drop=True)
    print(f"[{field_name:12s}] computing geometry for {len(df_clean):,} dipoles …", end=" ")

    geo = compute_observing_geometry(
        ra_deg=df_clean["r:ra"].values,
        dec_deg=df_clean["r:dec"].values,
        mjd=df_clean["r:midpointMjdTai"].values,
    )
    df_clean = pd.concat([df_clean, geo], axis=1)

    # Signed and folded dipole-parallactic angle differences
    if "r:dipoleAngle" in df_clean.columns:
        raw = df_clean["r:dipoleAngle"].values
        parang = df_clean["parallactic_angle_deg"].values
        diff = (raw - parang + 180.0) % 360.0 - 180.0
        df_clean["delta_dipole_para"] = diff
        adiff = np.abs(diff)
        df_clean["delta_dipole_para_folded"] = np.where(adiff <= 90.0, adiff, 180.0 - adiff)

    frames.append(df_clean)
    print("done")

if frames:
    df_all = pd.concat(frames, ignore_index=True)
    print(f"\nTotal dipoles with geometry: {len(df_all):,}")
    cols_show = [
        "field",
        "r:band",
        "r:midpointMjdTai",
        "r:dipoleAngle",
        "r:dipoleLength",
        "parallactic_angle_deg",
        "hour_angle_hr",
        "hour_angle_deg",
        "zenith_angle_deg",
        "sin_zenith",
        "azimuth_deg",
        "altitude_deg",
        "airmass",
        "delta_dipole_para",
        "delta_dipole_para_folded",
    ]
    display(df_all[[c for c in cols_show if c in df_all.columns]].describe())
else:
    df_all = pd.DataFrame()
    print("No dipoles found — nothing to analyse.")

## 5. Quick summary table — `r:dipoleLength` per DDF and band

Median, 16th–84th percentile, and min/max of `r:dipoleLength`
before plotting, to understand the dynamic range we are dealing with.

In [ ]:
if not df_all.empty and "r:dipoleLength" in df_all.columns:
    grp = df_all.groupby(["field", "r:band"])["r:dipoleLength"]
    summary = grp.agg(
        n="count",
        min=np.min,
        p16=lambda x: np.percentile(x.dropna(), 16),
        median=np.median,
        p84=lambda x: np.percentile(x.dropna(), 84),
        max=np.max,
    ).round(4)
    display(summary)
else:
    print("r:dipoleLength not available.")

## 6. 2×3 figure — Log-binned `r:dipoleLength` histograms per DDF, stacked by band

### Binning strategy

Because `r:dipoleLength` spans more than one order of magnitude, the bin edges
are spaced logarithmically:

```python
log_edges = np.logspace(np.log10(lo), np.log10(hi), N_BINS + 1)
```

The x-axis is displayed on a log scale.  Bands are stacked bottom-up
in the order u → g → r → i → z → y.

### Layout

```
┌─────────┬─────────┬─────────┐
│  COSMOS │  ECDFS  │ EDFS-a  │
├─────────┼─────────┼─────────┤
│  EDFS-b │  EDFS   │   M49   │
└─────────┴─────────┴─────────┘
```

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
N_BINS = 40  # number of log-spaced bins
SEP_COL = "r:dipoleLength"  # column to histogram

# Global range: compute from all available data
if not df_all.empty and SEP_COL in df_all.columns:
    vals_global = df_all[SEP_COL].dropna().values
    vals_global = vals_global[vals_global > 0]  # log requires > 0
    lo_global = vals_global.min()
    hi_global = vals_global.max()
    print(f"Global range of {SEP_COL}: [{lo_global:.4f}, {hi_global:.4f}]")
else:
    lo_global, hi_global = 0.01, 100.0
    print("No data — using fallback range.")

log_edges = np.logspace(np.log10(lo_global), np.log10(hi_global), N_BINS + 1)

In [ ]:
figname = "hist_dipoleLength_logbin_per_ddf"

nrows, ncols = 2, 3
fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(ncols * 4.5, nrows * 3.6),
    layout="constrained",
)

for idx, field_name in enumerate(DDF_NAMES):
    ax = axes[idx // ncols][idx % ncols]

    if df_all.empty or SEP_COL not in df_all.columns:
        ax.set_visible(False)
        continue

    sub = df_all[df_all["field"] == field_name]
    if sub.empty:
        ax.set_visible(False)
        continue

    # Determine bands present in this field, in canonical order
    bands_present = (
        [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()] if "r:band" in sub.columns else []
    )

    bottom = np.zeros(N_BINS)

    for band in bands_present:
        vals_b = sub.loc[sub["r:band"] == band, SEP_COL].dropna().values
        vals_b = vals_b[vals_b > 0]  # safety: log requires > 0
        cnts, _ = np.histogram(vals_b, bins=log_edges)
        ax.bar(
            log_edges[:-1],  # left edges
            cnts,
            width=np.diff(log_edges),  # variable width for log bins
            bottom=bottom,
            align="edge",
            color=BAND_COLORS.get(band, "grey"),
            alpha=0.85,
            edgecolor="white",
            linewidth=0.3,
            label=band,
        )
        bottom += cnts

    n_total = sub[SEP_COL].dropna().shape[0]

    ax.set_xscale("log")
    ax.set_xlabel(f"{SEP_COL}", fontsize=8)
    ax.set_ylabel("N dipoles", fontsize=8)
    ax.set_title(f"{field_name}  (n={n_total:,})", fontsize=9)
    ax.tick_params(labelsize=7)
    ax.set_xlim(lo_global, hi_global)

    # Mark the median (all bands combined)
    vals_field = sub[SEP_COL].dropna().values
    vals_field = vals_field[vals_field > 0]
    if len(vals_field) > 0:
        med = np.median(vals_field)
        ax.axvline(med, color="black", lw=1.2, ls="--", alpha=0.7, label=f"median={med:.3f}")
        ax.legend(fontsize=6, loc="upper right", framealpha=0.5)

# Hide any unused panels
for idx in range(len(DDF_NAMES), nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

# Shared band legend at the bottom
legend_handles = [mpatches.Patch(facecolor=BAND_COLORS[b], label=b) for b in BAND_ORDER if b in BAND_COLORS]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=6,
    fontsize=9,
    frameon=False,
    bbox_to_anchor=(0.5, -0.04),
)

fig.suptitle(
    "Dipole angular separation — log-binned histograms per DDF, stacked by band",
    y=1.01,
    fontsize=11,
)

savefig(figname)
plt.show()

## 7. Normalised version — density (probability) histograms

Same figure but each field is normalised to unit area (density=True
equivalent) so that different sample sizes do not dominate the visual
comparison.  The y-axis now shows probability density per log-unit.

In [ ]:
figname_norm = "hist_dipoleLength_logbin_per_ddf_normed"

fig2, axes2 = plt.subplots(
    nrows,
    ncols,
    figsize=(ncols * 4.5, nrows * 3.6),
    layout="constrained",
)

for idx, field_name in enumerate(DDF_NAMES):
    ax = axes2[idx // ncols][idx % ncols]

    if df_all.empty or SEP_COL not in df_all.columns:
        ax.set_visible(False)
        continue

    sub = df_all[df_all["field"] == field_name]
    if sub.empty:
        ax.set_visible(False)
        continue

    bands_present = (
        [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()] if "r:band" in sub.columns else []
    )

    n_total_field = sub[SEP_COL].dropna().shape[0]
    # Bin widths in log space (for density normalisation)
    log_widths = np.diff(log_edges)  # variable widths

    bottom = np.zeros(N_BINS)

    for band in bands_present:
        vals_b = sub.loc[sub["r:band"] == band, SEP_COL].dropna().values
        vals_b = vals_b[vals_b > 0]
        cnts, _ = np.histogram(vals_b, bins=log_edges)
        # Normalise: divide by total count × bin width so bars sum to 1
        density = cnts / (n_total_field * log_widths) if n_total_field > 0 else cnts
        ax.bar(
            log_edges[:-1],
            density,
            width=log_widths,
            bottom=bottom,
            align="edge",
            color=BAND_COLORS.get(band, "grey"),
            alpha=0.85,
            edgecolor="white",
            linewidth=0.3,
            label=band,
        )
        bottom += density

    vals_field = sub[SEP_COL].dropna().values
    vals_field = vals_field[vals_field > 0]
    if len(vals_field) > 0:
        med = np.median(vals_field)
        ax.axvline(med, color="black", lw=1.2, ls="--", alpha=0.7, label=f"median={med:.3f}")
        ax.legend(fontsize=6, loc="upper right", framealpha=0.5)

    ax.set_xscale("log")
    ax.set_xlabel(f"{SEP_COL}", fontsize=8)
    ax.set_ylabel("Probability density", fontsize=8)
    ax.set_title(f"{field_name}  (n={n_total_field:,})", fontsize=9)
    ax.tick_params(labelsize=7)
    ax.set_xlim(lo_global, hi_global)

for idx in range(len(DDF_NAMES), nrows * ncols):
    axes2[idx // ncols][idx % ncols].set_visible(False)

fig2.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=6,
    fontsize=9,
    frameon=False,
    bbox_to_anchor=(0.5, -0.04),
)
fig2.suptitle(
    "Dipole angular separation — normalised log histograms per DDF, stacked by band",
    y=1.01,
    fontsize=11,
)

savefig(figname_norm)
plt.show()

## 8. Overlay plot — median per DDF for comparison

Single panel overlaying the median separation (all bands) as vertical lines
for each DDF, on a shared log-binned histogram normalised to density.
Colour per DDF uses the `DEEP_FIELDS_COLORSTYLE` palette.

In [ ]:
DEEP_FIELDS_COLORSTYLE = {
    "COSMOS": {"color": "r"},
    "ECDFS": {"color": "grey"},
    "EDFS-a": {"color": "b"},
    "EDFS-b": {"color": "g"},
    "EDFS": {"color": "magenta"},
    "M49": {"color": "purple"},
}

figname_overlay = "hist_dipoleLength_overlay_ddf"

fig3, ax3 = plt.subplots(figsize=(6, 4), layout="constrained")

for field_name in DDF_NAMES:
    if df_all.empty or SEP_COL not in df_all.columns:
        continue
    sub = df_all[df_all["field"] == field_name]
    if sub.empty:
        continue
    vals_field = sub[SEP_COL].dropna().values
    vals_field = vals_field[vals_field > 0]
    if len(vals_field) < 3:
        continue
    n_f = len(vals_field)
    cnts, _ = np.histogram(vals_field, bins=log_edges)
    density = cnts / (n_f * np.diff(log_edges))
    color = DEEP_FIELDS_COLORSTYLE.get(field_name, {"color": "k"})["color"]
    centers = np.sqrt(log_edges[:-1] * log_edges[1:])  # geometric midpoints
    ax3.step(
        np.concatenate([[log_edges[0]], centers]),
        np.concatenate([[0], density]),
        where="pre",
        color=color,
        lw=1.5,
        label=field_name,
    )
    med = np.median(vals_field)
    ax3.axvline(med, color=color, lw=1.0, ls=":", alpha=0.8)

ax3.set_xscale("log")
ax3.set_xlabel(f"{SEP_COL}  [log scale]", fontsize=9)
ax3.set_ylabel("Probability density", fontsize=9)
ax3.set_title("Dipole separation — overlay of all DDFs (dotted = median)", fontsize=10)
ax3.legend(fontsize=8, loc="upper right")
ax3.set_xlim(lo_global, hi_global)

savefig(figname_overlay)
plt.show()

## 9. Per-band overlay for each DDF

For each DDF independently: step-histogram per band (normalised),
to see whether there is a band dependence within a field.

In [ ]:
figname_band = "hist_dipoleLength_perband_per_ddf"

fig4, axes4 = plt.subplots(
    nrows,
    ncols,
    figsize=(ncols * 4.5, nrows * 3.6),
    layout="constrained",
)

for idx, field_name in enumerate(DDF_NAMES):
    ax = axes4[idx // ncols][idx % ncols]

    if df_all.empty or SEP_COL not in df_all.columns:
        ax.set_visible(False)
        continue

    sub = df_all[df_all["field"] == field_name]
    if sub.empty:
        ax.set_visible(False)
        continue

    bands_present = (
        [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()] if "r:band" in sub.columns else []
    )
    n_total_field = sub[SEP_COL].dropna().shape[0]

    for band in bands_present:
        vals_b = sub.loc[sub["r:band"] == band, SEP_COL].dropna().values
        vals_b = vals_b[vals_b > 0]
        if len(vals_b) < 3:
            continue
        n_b = len(vals_b)
        cnts, _ = np.histogram(vals_b, bins=log_edges)
        density_b = cnts / (n_b * np.diff(log_edges))
        centers = np.sqrt(log_edges[:-1] * log_edges[1:])
        ax.step(
            np.concatenate([[log_edges[0]], centers]),
            np.concatenate([[0], density_b]),
            where="pre",
            color=BAND_COLORS.get(band, "grey"),
            lw=1.5,
            label=band,
        )
        med_b = np.median(vals_b)
        ax.axvline(med_b, color=BAND_COLORS.get(band, "grey"), lw=0.8, ls=":", alpha=0.7)

    ax.set_xscale("log")
    ax.set_xlabel(f"{SEP_COL}", fontsize=8)
    ax.set_ylabel("Probability density", fontsize=8)
    ax.set_title(f"{field_name}  (n={n_total_field:,})", fontsize=9)
    ax.tick_params(labelsize=7)
    ax.set_xlim(lo_global, hi_global)
    ax.legend(fontsize=7, loc="upper right", framealpha=0.5)

for idx in range(len(DDF_NAMES), nrows * ncols):
    axes4[idx // ncols][idx % ncols].set_visible(False)

fig4.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=6,
    fontsize=9,
    frameon=False,
    bbox_to_anchor=(0.5, -0.04),
)
fig4.suptitle(
    "Dipole separation — per-band normalised histograms per DDF (dotted = band median)",
    y=1.01,
    fontsize=11,
)

savefig(figname_band)
plt.show()

## 10. Summary statistics table

Medians and percentiles (16th, 84th) of `r:dipoleLength` per DDF
(all bands combined), sorted by median to confirm the physical
expectation: ECDFS smallest, COSMOS/M49 largest.

In [ ]:
if not df_all.empty and SEP_COL in df_all.columns:
    rows_summary = []
    for field_name in DDF_NAMES:
        sub = df_all[df_all["field"] == field_name]
        vals = sub[SEP_COL].dropna().values
        vals = vals[vals > 0]
        if len(vals) == 0:
            continue
        dec_field = DEEP_FIELDS[field_name][1]
        rows_summary.append(
            {
                "field": field_name,
                "dec_deg": dec_field,
                "n": len(vals),
                "min": round(vals.min(), 4),
                "p16": round(np.percentile(vals, 16), 4),
                "median": round(np.median(vals), 4),
                "p84": round(np.percentile(vals, 84), 4),
                "max": round(vals.max(), 4),
                "mean": round(vals.mean(), 4),
                "std": round(vals.std(), 4),
            }
        )
    df_summary = pd.DataFrame(rows_summary).sort_values("median")
    display(df_summary)
else:
    print("No data available for summary.")

## 11. Physical interpretation

| DDF | δ (deg) | Zenith distance at transit | Expected DCR | Expected separation |
|-----|---------|---------------------------|--------------|--------------------|
| ECDFS | −27.8 | ~2.4° (near zenith) | small | **small** |
| EDFS-a | −49.3 | ~19.1° | moderate | moderate |
| EDFS-b | −47.6 | ~17.4° | moderate | moderate |
| EDFS | −48.4 | ~18.2° | moderate | moderate |
| COSMOS | +2.2 | ~32.4° | large | **large** |
| M49 | +8.0 | ~38.2° | large | **large** |

The Rubin latitude is φ = −30.24°.  Zenith distance at transit z = |φ − δ|.
DCR amplitude ∝ tan z ≈ sin z for moderate angles.

If the histograms above show ECDFS skewed toward smaller separations and
COSMOS/M49 toward larger ones, this directly supports the DCR origin of
the dipole artefacts.


In [ ]:
# Compute transit zenith distance for each DDF
print(f"{'DDF':12s}  δ (deg)  z_transit (deg)  sin(z_transit)")
print("-" * 55)
for field_name, (ra, dec) in DEEP_FIELDS.items():
    z_transit = abs(RUBIN_LAT_DEG - dec)
    sinz_transit = np.sin(np.radians(z_transit))
    print(f"{field_name:12s}  {dec:+7.2f}  {z_transit:15.2f}  {sinz_transit:.4f}")